# Chain-of-Thought and Prompt Chaining

The full code of [Chain-of-Thought and Prompt Chaining](https://residual-stream.lvtan71.workers.dev/posts/chain-of-thought-and-prompt-chaining/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/chain-of-thought-and-prompt-chaining) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/chain-of-thought-and-prompt-chaining.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 29 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB). The checked rows sample at temperature 0.7 and vary by machine; the greedy rows match exactly.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 29 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.18.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "chain-of-thought-and-prompt-chaining"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/chain-of-thought-and-prompt-chaining


## `serial_steps.py`

40 serial arithmetic problems for each length k in 1, 2, 3, 4, 6, 8, solved six ways: one direct call, one reasoned call, a chain answering directly or reasoning first, and the direct chain checked by agreement or by an oracle (three seeds each). Writes every greedy chained step to `chain_steps.json` and reports which steps fail.

In [2]:
!{sys.executable} serial_steps.py

k = 1
   direct                                 100%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                 98%     1.0 calls per problem
   chained, reasoned                      100%     1.0 calls per problem
   checked by agreement  98%/98%/98% (3 seeds)     2.0 calls per problem
   checked by oracle     98%/98%/98% (3 seeds)     1.1 calls per problem


k = 2
   direct                                  18%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                 95%     2.0 calls per problem
   chained, reasoned                      100%     2.0 calls per problem
   checked by agreement  95%/95%/95% (3 seeds)     4.1 calls per problem
   checked by oracle     95%/95%/95% (3 seeds)     2.1 calls per problem


k = 3
   direct                                   2%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                100%     3.0 calls per problem
   chained, reasoned                       98%     3.0 calls per problem
   checked by agreement 100%/100%/100% (3 seeds)     6.0 calls per problem
   checked by oracle    100%/100%/100% (3 seeds)     3.0 calls per problem


k = 4
   direct                                   2%     1.0 calls per problem, 1 hit the token cap
   reasoned                                98%     1.0 calls per problem
   chained                                 92%     4.0 calls per problem
   chained, reasoned                       95%     4.0 calls per problem
   checked by agreement  92%/92%/92% (3 seeds)     8.0 calls per problem
   checked by oracle     95%/95%/95% (3 seeds)     4.2 calls per problem


k = 6
   direct                                   0%     1.0 calls per problem
   reasoned                                98%     1.0 calls per problem
   chained                                 80%     6.0 calls per problem, 2 hit the token cap
   chained, reasoned                       88%     6.0 calls per problem
   checked by agreement  82%/80%/80% (3 seeds)    12.5 calls per problem
   checked by oracle     88%/85%/82% (3 seeds)     6.5 calls per problem


k = 8
   direct                                   2%     1.0 calls per problem
   reasoned                                92%     1.0 calls per problem
   chained                                 70%     8.0 calls per problem, 3 hit the token cap
   chained, reasoned                       82%     8.0 calls per problem
   checked by agreement  75%/75%/72% (3 seeds)    16.7 calls per problem
   checked by oracle     85%/88%/75% (3 seeds)     8.8 calls per problem

one greedy chained step right: 97.1% overall; multiply 98.7%, add or subtract 96.3%; starting from a number of 1,000 or more 81.6% (49 steps)


The post printed:

```text
k = 1
   direct                                 100%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                 98%     1.0 calls per problem
   chained, reasoned                      100%     1.0 calls per problem
   checked by agreement  98%/98%/98% (3 seeds)     2.0 calls per problem
   checked by oracle     98%/98%/98% (3 seeds)     1.1 calls per problem
k = 2
   direct                                  18%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                 95%     2.0 calls per problem
   chained, reasoned                      100%     2.0 calls per problem
   checked by agreement  95%/95%/98% (3 seeds)     4.1 calls per problem
   checked by oracle     98%/98%/95% (3 seeds)     2.1 calls per problem
k = 3
   direct                                   2%     1.0 calls per problem
   reasoned                               100%     1.0 calls per problem
   chained                                100%     3.0 calls per problem
   chained, reasoned                       98%     3.0 calls per problem
   checked by agreement 100%/100%/100% (3 seeds)     6.0 calls per problem
   checked by oracle    100%/100%/100% (3 seeds)     3.0 calls per problem
k = 4
   direct                                   2%     1.0 calls per problem, 1 hit the token cap
   reasoned                                98%     1.0 calls per problem
   chained                                 92%     4.0 calls per problem
   chained, reasoned                       95%     4.0 calls per problem
   checked by agreement  98%/95%/95% (3 seeds)     8.0 calls per problem
   checked by oracle     98%/95%/95% (3 seeds)     4.2 calls per problem
k = 6
   direct                                   0%     1.0 calls per problem
   reasoned                                98%     1.0 calls per problem
   chained                                 80%     6.0 calls per problem, 2 hit the token cap
   chained, reasoned                       88%     6.0 calls per problem
   checked by agreement  78%/78%/80% (3 seeds)    12.4 calls per problem
   checked by oracle     82%/85%/85% (3 seeds)     6.5 calls per problem
k = 8
   direct                                   2%     1.0 calls per problem
   reasoned                                92%     1.0 calls per problem
   chained                                 70%     8.0 calls per problem, 3 hit the token cap
   chained, reasoned                       82%     8.0 calls per problem
   checked by agreement  72%/68%/72% (3 seeds)    16.8 calls per problem
   checked by oracle     72%/80%/82% (3 seeds)     8.8 calls per problem

one greedy chained step right: 97.1% overall; multiply 98.7%, add or subtract 96.3%; starting from a number of 1,000 or more 81.6% (49 steps)
```

## `step_errors.py`

Reads `chain_steps.json` and prints the error analysis: wrong steps, how many were multiplications or started from 1,000 or more, accuracy below 1,000, wrong steps around zero, and per-step accuracy on the eight-step problems.

In [3]:
!{sys.executable} step_errors.py

960 28 wrong; multiply 4 ; from >=1000 9
below 1000 right 0.9791437980241493
negatives or crossing zero among wrong: 7
last 320 (k=8): 0.95625


The post printed:

```text
960 28 wrong; multiply 4 ; from >=1000 9
below 1000 right 0.9791437980241493
negatives or crossing zero among wrong: 7
last 320 (k=8): 0.95625
```